# Q1. Classification (shared baseline)

This notebook builds **the classifier** used as the reference point by every other section of the project.
All the logic lives in `common.py`, so the other sections import the same preprocessing, split and model instead of copying code.

**Files**
* `common.py`: shared module (loading, preprocessing, split, model, evaluation)
* `data/adult.data`: download it from UCI and put it in a `data/` folder next to `common.py`

In [1]:
import numpy as np
import pandas as pd
import common as c

pd.set_option("display.precision", 4)
print("seed:", c.SEED, "| age threshold:", c.AGE_THRESHOLD, "| model:", c.MODEL, "| C:", c.LOGREG_C)

seed: 42 | age threshold: 37 | model: logreg | C: 1.0


## 1. Raw data

In [2]:
raw = c.load_raw()
print(raw.shape)
print("missing values per column:")
print(raw.isna().sum()[raw.isna().sum() > 0])
raw.head()

(32561, 15)
missing values per column:
workclass         1836
occupation        1843
native-country     583
dtype: int64


,age,workclass,fnlwgt,education,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country,income
0,39,State-gov,77516,Bachelors,13,Never-married,Adm-clerical,Not-in-family,White,Male,2174,0,40,United-States,<=50K
1,50,Self-emp-not-inc,83311,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,13,United-States,<=50K
2,38,Private,215646,HS-grad,9,Divorced,Handlers-cleaners,Not-in-family,White,Male,0,0,40,United-States,<=50K
3,53,Private,234721,11th,7,Married-civ-spouse,Handlers-cleaners,Husband,Black,Male,0,0,40,United-States,<=50K
4,28,Private,338409,Bachelors,13,Married-civ-spouse,Prof-specialty,Wife,Black,Female,0,0,40,Cuba,<=50K


## 2. Preprocessing

Choices made in `common.preprocess()`:

* **Missing values** (`?` in workclass, occupation, native-country): the 2,399 incomplete rows (7.4%) are dropped, as AIF360 does for Adult. 30,162 rows remain.
* **Dropped columns**: `fnlwgt` is a census sampling weight, not a property of the person; `education` is redundant with `education-num` (one-to-one mapping).
* **Age** is binarized (see threshold choice below): `age = 1` if age >= threshold, else 0.
* **Sex** is encoded `1 = Male`, `0 = Female`.
* **Label** `income`: `1 = ">50K"`, `0 = "<=50K"`.
* Encoding happens inside the model pipeline (fitted on the training set only, so no information leaks from validation/test): one-hot encoding for categorical columns, standardization for numerical columns, age and sex passed through as 0/1.

### Choosing the age threshold

In [3]:
clean = raw.dropna()
y = clean["income"] == ">50K"
rows = []
for t in [25, 30, 37, 40]:
    old = clean["age"] >= t
    rows.append({
        "threshold": t,
        "share young (< t)": (~old).mean(),
        "P(>50K | young)": y[~old].mean(),
        "P(>50K | old)": y[old].mean(),
        "young with >50K (count)": int(y[~old].sum()),
    })
print("median age:", clean["age"].median())
pd.DataFrame(rows)

median age: 37.0


,threshold,share young (< t),P(>50K | young),P(>50K | old),young with >50K (count)
0,25,0.1614,0.0119,0.2945,58
1,30,0.2912,0.0562,0.3281,494
2,37,0.4837,0.1307,0.3597,1907
3,40,0.5635,0.1610,0.3624,2736


We use **37, the median age**. It gives two groups of almost equal size, which:
* keeps fairness metrics stable on the test set (with 25, only 58 young people in the whole data earn >50K, so about 12 in the test set, which makes recall-based metrics very noisy);
* gives the best relative accuracy for the LDP count estimates in Q4 (no tiny cell in the Age x Sex cross-tabulation);
* still shows a strong disparity (13% vs 36% of >50K), so the fairness study remains meaningful.

The threshold is a constant in `common.py` (`AGE_THRESHOLD`) if the group prefers another value.

In [4]:
df = c.load_data()
print(df.shape)
print("group sizes:")
print(df["age"].value_counts().rename({0: "young", 1: "old"}).to_dict(),
      df["sex"].value_counts().rename({0: "female", 1: "male"}).to_dict())
df.head()

(30162, 13)
group sizes:
{'old': 15572, 'young': 14590} {'male': 20380, 'female': 9782}


,age,sex,workclass,marital-status,occupation,relationship,race,native-country,education-num,capital-gain,capital-loss,hours-per-week,income
0,1,1,State-gov,Never-married,Adm-clerical,Not-in-family,White,United-States,13,2174,0,40,0
1,1,1,Self-emp-not-inc,Married-civ-spouse,Exec-managerial,Husband,White,United-States,13,0,0,13,0
2,1,1,Private,Divorced,Handlers-cleaners,Not-in-family,White,United-States,9,0,0,40,0
3,1,1,Private,Married-civ-spouse,Handlers-cleaners,Husband,Black,United-States,7,0,0,40,0
4,0,0,Private,Married-civ-spouse,Prof-specialty,Wife,Black,Cuba,13,0,0,40,0


## 3. Train / validation / test split

60/20/20, stratified on the label, fixed seed. The split is stratified **on the label only**: a private copy of the data (Age and Sex replaced by noisy values, same row order) therefore gets exactly the same rows in each set, which is what Q4 asks ("split the private data in the same manner").

In [5]:
train, val, test = c.split_data(df)
pd.DataFrame({
    "rows": [len(train), len(val), len(test)],
    "share >50K": [train.income.mean(), val.income.mean(), test.income.mean()],
}, index=["train", "validation", "test"])

,rows,share >50K
train,18096,0.2489
validation,6033,0.2490
test,6033,0.2490


In [6]:
# Check: randomizing age/sex (as LDP will do) does not change the split
noisy = df.copy()
rng = np.random.default_rng(0)
noisy[["age", "sex"]] = rng.integers(0, 2, size=(len(noisy), 2))
same = all((a == b).all() for a, b in zip(c.split_indices(df), c.split_indices(noisy)))
print("same split on noisy data:", same)

same split on noisy data: True


## 4. Model selection on the validation set

The validation set is used to choose the model and its regularization strength `C`; the test set is only used once, at the end.

In [7]:
rows = []
for C_ in [0.01, 0.1, 1, 10, 100]:
    m = c.train_classifier(train, model="logreg", C=C_)
    rows.append({"model": "logistic regression", "C": C_, **c.evaluate(m, val)})
m = c.train_classifier(train, model="hgb")
rows.append({"model": "gradient boosting (HGB)", "C": None, **c.evaluate(m, val)})
pd.DataFrame(rows)

,model,C,accuracy,balanced_accuracy,precision,recall,f1,roc_auc
0,logistic regression,0.01,0.8480,0.7550,0.7595,0.5699,0.6512,0.9050
1,logistic regression,0.10,0.8521,0.7718,0.7484,0.6119,0.6733,0.9087
2,logistic regression,1.00,0.8543,0.7775,0.7486,0.6245,0.6809,0.9090
3,logistic regression,10.00,0.8531,0.7767,0.7444,0.6245,0.6792,0.9091
4,logistic regression,100.00,0.8531,0.7772,0.7437,0.6258,0.6797,0.9091
5,gradient boosting (HGB),NaN,0.8676,0.7943,0.7823,0.6485,0.7091,0.9239


**Choice: logistic regression with C = 1** (best validation F1 and accuracy among the regularization values).

Gradient boosting is about 1.5 points more accurate, but logistic regression fits the rest of the project better:
* **Q2 (sustainability)**: training is repeated 100 times; a light model keeps measurements quick and simple to interpret.
* **Q3 / Q5 (fairness)**: it accepts the instance weights produced by AIF360 Reweighing (`sample_weight`).
* **Q6 (explainability)**: its probabilities are reasonably calibrated, so "wrong but highly confident" is meaningful, and its explanations (coefficients, exact linear SHAP values) are easy to read.
* The reweighing demo of the AIF360 practice session (`demo_reweighing_preproc`) also uses a logistic regression.

Switching to gradient boosting only requires `MODEL = "hgb"` in `common.py`.

## 5. The classifier: performance on the test set

In [8]:
clf = c.train_classifier(train)
test_metrics = c.evaluate(clf, test)
pd.Series(test_metrics, name="test")

accuracy             0.8495
balanced_accuracy    0.7683
precision            0.7419
recall               0.6065
f1                   0.6674
roc_auc              0.9033
Name: test, dtype: float64

In [9]:
X_test, y_test = c.X_y(test)
pd.crosstab(y_test.rename("actual"), pd.Series(clf.predict(X_test), index=test.index, name="predicted"))

predicted,0,1
actual,,
0,4214,317
1,591,911


### Disaggregated performance (preview for Q3)
Same metrics, per group of each protected attribute (1 = older / male).

In [10]:
c.evaluate_by_group(clf, test)

,attribute,value,n,base_rate,positive_rate,accuracy,recall (TPR)
0,age,0,2985,0.1374,0.0797,0.9002,0.4268
1,age,1,3048,0.3583,0.3248,0.7999,0.6740
2,sex,0,1968,0.1194,0.0884,0.9212,0.5404
3,sex,1,4065,0.3117,0.2593,0.8148,0.6188


**Observations**
* The classifier reaches about **85% accuracy** (F1 0.67, ROC AUC 0.90). Accuracy alone is flattering because the classes are imbalanced (25% >50K; always predicting "<=50K" would already give 75%), so balanced accuracy and F1 are also reported.
* The model predicts ">50K" for 8% of young people vs 33% of older people, and 9% of women vs 26% of men. The model under-predicts ">50K" for everyone, but relatively more for the unprivileged groups: the ratio of positive rates (unprivileged / privileged) is 0.25 for age and 0.34 for sex, while it is 0.38 for both in the true labels. Recall (TPR) is also much lower for young people (0.43 vs 0.67) and for women (0.54 vs 0.62). This is the starting point of the fairness analysis in Q3.

<!--
## 6. How to reuse this in the other sections

```python
import common as c

df = c.load_data()                       # preprocessed data, binary age/sex
train, val, test = c.split_data(df)      # always the same split
clf = c.train_classifier(train)          # "the classifier"
c.evaluate(clf, test)                    # performance metrics
c.evaluate_by_group(clf, test)           # per-group metrics
```

* **Q2**: time/measure `c.train_classifier(train)` (training) and `clf.predict(X_test)` (inference) inside the 100-iteration loops.
* **Q3**: fair classifier with reweighing: `c.train_classifier(train, sample_weight=w)` where `w` are the AIF360 instance weights for the training rows.
* **Q4**: build `private = df.copy()`, replace `private["age"]` and `private["sex"]` with the randomized-response values (keep the row order), then `c.split_data(private)` gives the same rows as for `df`.
* **Q5 / Q6**: to audit with the real values, evaluate on `test` (real age/sex) the predictions made from `private_test` (same rows, same index).
* **Q6**: `c.feature_names(clf)` gives the names of the encoded features for SHAP/LIME.
-->

## 6. Notes for the report (Q1)

* **Data**: `adult.data` from UCI (32,561 records). Rows with missing values removed (30,162 left). `fnlwgt` and `education` dropped.
* **Binarization**: Age split at the median (37): young < 37, older >= 37. Sex: male / female. In both cases, 1 denotes the group with the higher rate of income >50K.
* **Split**: 60% train (18,096), 20% validation (6,033), 20% test (6,033), stratified on the label, seed 42.
* **Model**: one-hot encoding + standardization, logistic regression (C = 1, chosen on validation among 0.01 to 100; gradient boosting considered and set aside for the reasons above).
* **Test performance**: accuracy 0.849, balanced accuracy 0.768, precision 0.742, recall 0.607, F1 0.667, ROC AUC 0.903.